# 4. Reconhecimento científico, repescagem, disponibilidade e leituras (A6, A7, A8)

Sequência do ProKnow-C: corte por citações → leitura do resumo dos mais citados (grupo A) →
repescagem → leitura do resumo dos repescados → disponibilidade → leitura do texto completo.

In [ ]:
#@title Preparação (rode esta célula primeiro) { display-mode: "form" }
REPOSITORIO = "https://github.com/SEU_USUARIO/curso_proknowc.git"
import os, subprocess, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/curso_proknowc"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPOSITORIO, "/content/curso_proknowc"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "/content/curso_proknowc"], check=True)
    sys.path.insert(0, "/content/curso_proknowc/src")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from curso_proknowc import colab

pastas = colab.preparar()
print("Tudo pronto. Pasta do curso:", pastas.raiz)

In [ ]:
#@title Entrada { display-mode: "form" }
#@markdown **exemplo do curso**: começa do resultado já pronto do exemplo (previsão de velocidade do vento).
#@markdown **meus arquivos**: começa do resultado que você salvou no notebook anterior.
#@markdown **EMAIL**: o seu e-mail, usado nas consultas ao OpenAlex e ao Unpaywall (o Unpaywall exige).
FONTE = "exemplo do curso" #@param ["exemplo do curso", "meus arquivos"]
EMAIL = "" #@param {type:"string"}
alinhados = colab.carregar(pastas, FONTE, "03_alinhados_titulo", notebook="04")
print(f"{len(alinhados)} artigos carregados.")

## Reconhecimento científico (A6)

As citações vêm de uma fonte única, o OpenAlex, para que os números sejam comparáveis entre si.

In [ ]:
#@title Citações { display-mode: "form" }
#@markdown Consulta o OpenAlex (precisa de internet). O exemplo do curso já vem com as citações consultadas.
ATUALIZAR_CITACOES = True #@param {type:"boolean"}
from curso_proknowc import reconhecimento
if ATUALIZAR_CITACOES:
    alinhados, avisos = reconhecimento.atualizar_citacoes(alinhados, email=EMAIL or None)
    for aviso in avisos:
        print("•", aviso)
print("Quantos artigos ficam acima de cada corte:")
display(reconhecimento.artigos_por_corte(alinhados))

In [ ]:
#@title Corte por citações { display-mode: "form" }
#@markdown Percentual das citações acumuladas a partir do qual os artigos ficam "abaixo do corte". A escolha é sua (a literatura costuma usar de 80 a 85%).
CORTE = 80 #@param {type:"slider", min:50, max:95, step:5}
reconhecimento.grafico_curva(alinhados, CORTE)
plt.show()
acima, abaixo = reconhecimento.dividir_por_corte(alinhados, CORTE)
acima = acima.assign(grupo="A")
print(f"Acima do corte: {len(acima)} artigos. Abaixo do corte: {len(abaixo)}.")

In [ ]:
#@title Leitura do resumo: artigos acima do corte { display-mode: "form" }
from curso_proknowc import triagem
decisoes = colab.arquivo_decisoes(pastas, FONTE)
display(triagem.interface(acima, "resumo", decisoes))

## Repescagem (A7)

Rode depois de terminar a leitura acima. Recentes voltam para a leitura do resumo; os antigos só voltam se tiverem um autor do grupo A.

In [ ]:
#@title Repescagem { display-mode: "form" }
ANO_DA_PESQUISA = 2026 #@param {type:"integer"}
from curso_proknowc import repescagem
try:
    grupo_a, _ = colab.aplicar_triagem(acima, "resumo", decisoes)
except ValueError as erro:
    raise SystemExit(f"{erro} (leitura do resumo dos artigos acima do corte)")
enviados, descartados = repescagem.repescar(abaixo, grupo_a, ANO_DA_PESQUISA)
colab.registrar_etapa(pastas, "04", "Reconhecimento científico e repescagem", len(alinhados), descartados)
print(f"Grupo A: {len(grupo_a)} artigos. Repescados para a leitura do resumo: {len(enviados)}. Excluídos: {len(descartados)}.")
print("Autores do grupo A (banco de autores):")
display(repescagem.banco_autores(grupo_a).head(15))
if len(enviados):
    display(enviados[["id", "titulo", "ano", "motivo_repescagem"]])
else:
    print("Nenhum artigo abaixo do corte foi repescado: a próxima leitura pode ser pulada.")

In [ ]:
#@title Leitura do resumo: artigos repescados { display-mode: "form" }
display(triagem.interface(enviados, "resumo", decisoes))

In [ ]:
#@title Resultado da leitura dos resumos (rode depois de terminar) { display-mode: "form" }
para_resumo = pd.concat([acima, enviados], ignore_index=True)
try:
    aprovados, rejeitados = colab.aplicar_triagem(para_resumo, "resumo", decisoes)
except ValueError as erro:
    raise SystemExit(str(erro))
colab.registrar_etapa(pastas, "04", "Leitura do resumo", len(para_resumo), rejeitados)
print(f"Aprovados na leitura do resumo: {len(aprovados)}. Não alinhados: {len(rejeitados)}.")

## Disponibilidade do texto completo (A8)

Os artigos fechados ficam marcados para você buscar pelo Portal de Periódicos CAPES.

In [ ]:
#@title Disponibilidade { display-mode: "form" }
from curso_proknowc.disponibilidade import verificar_disponibilidade
if EMAIL.strip():
    aprovados = verificar_disponibilidade(aprovados, EMAIL.strip())
    display(aprovados[["id", "titulo", "acesso_aberto", "url_pdf", "obs_disponibilidade"]])
else:
    print("Preencha o EMAIL na célula de entrada e rode de novo para consultar o Unpaywall (ele exige um e-mail).")

## Leitura do texto completo

Leia cada artigo completo. Se não conseguir o texto, rejeite e escreva o motivo (ex.: "texto indisponível").

In [ ]:
#@title Leitura do texto completo { display-mode: "form" }
display(triagem.interface(aprovados, "texto", decisoes))

In [ ]:
#@title Aplicar as decisões e salvar o portfólio (rode depois de terminar) { display-mode: "form" }
BAIXAR_CSV = False #@param {type:"boolean"}
try:
    portfolio, rejeitados_texto = colab.aplicar_triagem(aprovados, "texto", decisoes)
except ValueError as erro:
    print(erro)
else:
    colab.registrar_etapa(pastas, "04", "Leitura do texto completo", len(aprovados), rejeitados_texto)
    colab.salvar(pastas, portfolio, "04_portfolio", baixar=BAIXAR_CSV)
    print(f"Portfólio: {len(portfolio)} artigos. Resultado salvo.")